<a href="https://colab.research.google.com/github/navya039/supernan-ai-dubbing/blob/main/notebooks/colab_pipeline_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supernan AI Dubbing Pipeline — Complete Version
**Runtime: T4 GPU required**

Run Cell 1, wait for it to finish, then run Cell 2 to download.

Pipeline: FFmpeg → Whisper → NLLB → gTTS → Wav2Lip

In [ ]:
# ============================================================
# COMPLETE SUPERNAN AI DUBBING PIPELINE
# ============================================================

# --- MOUNT DRIVE ---
from google.colab import drive
drive.mount('/content/drive')

import os, json, gc, subprocess
PROJECT_DIR = '/content/drive/MyDrive/supernan'
os.makedirs(PROJECT_DIR, exist_ok=True)
print('✅ Drive mounted!')

# --- INSTALL DEPENDENCIES ---
print('Installing dependencies...')
os.system('pip install openai-whisper gTTS soundfile librosa gdown transformers sentencepiece sacremoses -q')
os.system('apt-get install -y ffmpeg -q')
print('✅ Dependencies installed!')

# --- PATHS ---
VIDEO_PATH        = f'{PROJECT_DIR}/supernan_video.mp4'
CLIP_PATH         = f'{PROJECT_DIR}/clip_15s.mp4'
CLIP_AUDIO_PATH   = f'{PROJECT_DIR}/clip_audio.wav'
HINDI_MP3_PATH    = f'{PROJECT_DIR}/hindi_audio.mp3'
HINDI_WAV_PATH    = f'{PROJECT_DIR}/hindi_audio.wav'
HINDI_ALIGNED_PATH = f'{PROJECT_DIR}/hindi_audio_aligned.wav'
OUTPUT_PATH       = f'{PROJECT_DIR}/supernan_hindi_dubbed_final.mp4'

# ============================================================
# STAGE 1: DOWNLOAD VIDEO
# ============================================================
if not os.path.exists(VIDEO_PATH):
    print('⬇️ Downloading source video...')
    os.system(f'gdown "1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW" -O "{VIDEO_PATH}"')
else:
    print('✅ Video already in Drive, skipping download.')
print(f'📹 Video size: {os.path.getsize(VIDEO_PATH)/1024/1024:.1f} MB')

# ============================================================
# STAGE 2: EXTRACT 15 SECOND CLIP AND AUDIO
# ============================================================
print('\n🎬 Extracting 15s clip...')
os.system(f'ffmpeg -i "{VIDEO_PATH}" -ss 00:00:15 -to 00:00:30 -c:v libx264 -c:a aac "{CLIP_PATH}" -y -loglevel error')
os.system(f'ffmpeg -i "{CLIP_PATH}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{CLIP_AUDIO_PATH}" -y -loglevel error')
print(f'✅ Clip: {os.path.getsize(CLIP_PATH)/1024/1024:.1f} MB')
print(f'✅ Audio: {os.path.getsize(CLIP_AUDIO_PATH)/1024:.1f} KB')

# ============================================================
# STAGE 3: TRANSCRIBE WITH WHISPER
# ============================================================
print('\n🎙️ Loading Whisper medium...')
import whisper
whisper_model = whisper.load_model('medium')
result = whisper_model.transcribe(
    CLIP_AUDIO_PATH,
    word_timestamps=True,
    verbose=False
)
raw_text = result['text'].strip()
print(f'🌐 Detected language: {result["language"]}')
print(f'📝 Transcript: {raw_text}')

with open(f'{PROJECT_DIR}/transcript.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print('✅ Transcript saved!')

# Free GPU memory before loading next model
import torch
del whisper_model
gc.collect()
torch.cuda.empty_cache()
print('✅ GPU memory cleared!')

# ============================================================
# STAGE 4: TRANSLATE TO HINDI
# ============================================================
print('\n🌐 Loading NLLB translation model...')
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
trans_tokenizer = AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M')
trans_model = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M')

trans_tokenizer.src_lang = 'eng_Latn'
inputs = trans_tokenizer(
    raw_text,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=512
)

with torch.no_grad():
    translated = trans_model.generate(
        **inputs,
        forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids('hin_Deva'),
        max_length=512,
        num_beams=5
    )

hindi_final = trans_tokenizer.decode(translated[0], skip_special_tokens=True)
print(f'🇮🇳 Hindi: {hindi_final}')

with open(f'{PROJECT_DIR}/translation_hindi.txt', 'w', encoding='utf-8') as f:
    f.write(hindi_final)
print('✅ Translation saved!')

# Free GPU memory
del trans_model, trans_tokenizer
gc.collect()
torch.cuda.empty_cache()
print('✅ GPU memory cleared!')

# ============================================================
# STAGE 5: GENERATE HINDI AUDIO
# ============================================================
print('\n🔊 Generating Hindi audio...')
from gtts import gTTS
tts = gTTS(text=hindi_final, lang='hi', slow=False)
tts.save(HINDI_MP3_PATH)
os.system(f'ffmpeg -i "{HINDI_MP3_PATH}" -ar 16000 -ac 1 "{HINDI_WAV_PATH}" -y -loglevel error')
print(f'✅ Hindi audio: {os.path.getsize(HINDI_WAV_PATH)/1024:.1f} KB')

# ============================================================
# STAGE 6: ALIGN AUDIO DURATION
# ============================================================
print('\n⏱️ Aligning audio duration...')
import numpy as np
import soundfile as sf
import librosa

original, sr_orig = librosa.load(CLIP_AUDIO_PATH, sr=16000)
hindi, sr_hindi   = librosa.load(HINDI_WAV_PATH, sr=16000)

orig_duration  = len(original) / sr_orig
hindi_duration = len(hindi) / sr_hindi
print(f'Original: {orig_duration:.2f}s | Hindi: {hindi_duration:.2f}s')

diff = orig_duration - hindi_duration
if 0 < diff <= 2.0:
    ratio = orig_duration / hindi_duration
    hindi_aligned = librosa.effects.time_stretch(hindi, rate=ratio)
    print(f'🔧 Gently stretched by ratio: {ratio:.3f}')
elif diff > 2.0:
    padding = np.zeros(int(diff * sr_hindi))
    hindi_aligned = np.concatenate([hindi, padding])
    print(f'🔇 Padded with {diff:.2f}s silence')
else:
    hindi_aligned = hindi[:len(original)]
    print('✂️ Trimmed to fit')

sf.write(HINDI_ALIGNED_PATH, hindi_aligned, sr_hindi)
print(f'✅ Aligned audio saved!')

# ============================================================
# STAGE 7: SETUP WAV2LIP
# ============================================================
print('\n🔧 Setting up Wav2Lip...')

if not os.path.exists('/content/Wav2Lip'):
    os.system('git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip')
    print('✅ Wav2Lip cloned!')
else:
    print('✅ Wav2Lip already exists!')

os.makedirs('/content/Wav2Lip/checkpoints', exist_ok=True)

model_path = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'
if not os.path.exists(model_path):
    print('Downloading Wav2Lip model...')
    os.system(f'wget -q "https://github.com/Rudrabha/Wav2Lip/releases/download/models/wav2lip_gan.pth" -O "{model_path}"')
    print('✅ Model downloaded!')
else:
    print('✅ Model already exists!')

# ============================================================
# STAGE 8: PATCH WAV2LIP FOR NUMPY 2.X COMPATIBILITY
# ============================================================
print('\n🔧 Patching Wav2Lip for NumPy compatibility...')

patch_file = '/content/Wav2Lip/face_detection/detection/sfd/detect.py'
with open(patch_file, 'r') as f:
    content = f.read()

content = content.replace(
    'imgs = torch.from_numpy(imgs).float().to(device)',
    'imgs = torch.tensor(imgs.copy(), dtype=torch.float32).to(device)'
)

with open(patch_file, 'w') as f:
    f.write(content)
print('✅ Patched successfully!')

# ============================================================
# STAGE 9: RUN LIP SYNC
# ============================================================
print('\n🎬 Running lip sync — this takes 10-20 minutes...')
os.chdir('/content/Wav2Lip')

cmd = f'''python inference.py \
    --checkpoint_path checkpoints/wav2lip_gan.pth \
    --face "{CLIP_PATH}" \
    --audio "{HINDI_ALIGNED_PATH}" \
    --outfile "{OUTPUT_PATH}" \
    --resize_factor 2 \
    --pads 0 20 0 0'''

os.system(cmd)

# ============================================================
# FINAL CHECK
# ============================================================
if os.path.exists(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH)/1024/1024
    print(f'\n✅ FINAL VIDEO READY: {size:.1f} MB')
    print(f'📁 Saved to: {OUTPUT_PATH}')
    print('\n👉 Run Cell 2 below to download!')
else:
    print('\n❌ Output not found — check errors above')

Mounted at /content/drive
✅ Drive mounted!
Installing dependencies...
✅ Dependencies installed!
✅ Video already in Drive, skipping download.
📹 Video size: 593.1 MB

🎬 Extracting 15s clip...
✅ Clip: 7.6 MB
✅ Audio: 469.4 KB

🎙️ Loading Whisper medium...


100%|██████████████████████████████████████| 1.42G/1.42G [00:07<00:00, 212MiB/s]


Detected language: Kannada


  0%|          | 0/1501 [00:00<?, ?frames/s]

In [3]:
# CELL 2: Download final video
from google.colab import files
OUTPUT_PATH = '/content/drive/MyDrive/supernan/supernan_hindi_dubbed_final.mp4'
files.download(OUTPUT_PATH)
print('✅ Download started!')

FileNotFoundError: Cannot find file: /content/drive/MyDrive/supernan/supernan_hindi_dubbed_final.mp4

In [4]:
import os

# Check if output exists anywhere
print('Checking for output files...')
for f in os.listdir('/content/drive/MyDrive/supernan'):
    path = f'/content/drive/MyDrive/supernan/{f}'
    size = os.path.getsize(path)/1024/1024
    print(f'  {f}: {size:.1f} MB')

# Also check Wav2Lip directory
print('\nChecking Wav2Lip results folder...')
results_dir = '/content/Wav2Lip/results'
if os.path.exists(results_dir):
    for f in os.listdir(results_dir):
        print(f'  {f}')
else:
    print('No results folder found')

Checking for output files...
  supernan_video.mp4: 593.1 MB
  transcript.json: 0.0 MB
  transcript_raw.txt: 0.0 MB
  clip_15s.mp4: 7.6 MB
  clip_audio.wav: 0.5 MB
  translation_hindi.txt: 0.0 MB
  hindi_audio.wav: 0.2 MB
  hindi_audio.mp3: 0.1 MB
  hindi_audio_aligned.wav: 0.5 MB

Checking Wav2Lip results folder...
No results folder found


In [5]:
import os
os.chdir('/content/Wav2Lip')

PROJECT_DIR = '/content/drive/MyDrive/supernan'
CLIP_PATH = f'{PROJECT_DIR}/clip_15s.mp4'
HINDI_ALIGNED_PATH = f'{PROJECT_DIR}/hindi_audio_aligned.wav'
OUTPUT_PATH = f'{PROJECT_DIR}/supernan_hindi_dubbed_final.mp4'

!python inference.py \
    --checkpoint_path checkpoints/wav2lip_gan.pth \
    --face "{CLIP_PATH}" \
    --audio "{HINDI_ALIGNED_PATH}" \
    --outfile "{OUTPUT_PATH}" \
    --resize_factor 2 \
    --pads 0 20 0 0 2>&1

FileNotFoundError: [Errno 2] No such file or directory: '/content/Wav2Lip'

In [7]:
!pip install librosa==0.9.2 -q
print('✅ Librosa fixed!')




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.3/214.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.9 MB/s eta 0:00:00
✅ Librosa fixed!
fatal: destination path '/content/Wav2Lip' already exists and is not an empty directory.
✅ Model downloaded!
✅ Patched!
Using cuda for inference.
Reading video frames...
Number of frames available for inference: 900
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 1202)
Length of mel chunks: 892
  0% 0/7 [00:00<?, ?it/s]Downloading: "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" to /root/.cache/torch/hub/checkpoints/s3fd-619a316812.pth

  0% 0.00/85.7M [00:00<?, ?B/s]
  0% 128k/85.7M [00:00<03:41, 405kB/s]
  0% 256k/85.7M [00:00<02:36, 574kB/s]
  1% 512k/85.7M [00:00<01:32, 965kB/s]
  1% 896k/85.

In [8]:
import os

# Remove corrupted model and redownload
os.remove('/content/Wav2Lip/checkpoints/wav2lip_gan.pth')

# Use alternative download link
!wget "https://iiitaphyd-my.sharepoint.com/:u:/g/personal/radrabha_m_research_iiit_ac_in/Eb3LEzbfuKlJiR600lT4-PBBypp2wasTNeKMaBQCNeHEwg?e=ZRPG80&download=1" -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth

size = os.path.getsize('/content/Wav2Lip/checkpoints/wav2lip_gan.pth')/1024/1024
print(f'✅ Model downloaded: {size:.1f} MB')
# Should be around 415 MB

--2026-03-08 14:49:49--  https://iiitaphyd-my.sharepoint.com/:u:/g/personal/radrabha_m_research_iiit_ac_in/Eb3LEzbfuKlJiR600lT4-PBBypp2wasTNeKMaBQCNeHEwg?e=ZRPG80&download=1
Resolving iiitaphyd-my.sharepoint.com (iiitaphyd-my.sharepoint.com)... 13.107.136.10, 13.107.138.10, 2620:1ec:8f8::10, ...
Connecting to iiitaphyd-my.sharepoint.com (iiitaphyd-my.sharepoint.com)|13.107.136.10|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-08 14:49:52 ERROR 404: Not Found.

✅ Model downloaded: 0.0 MB


In [9]:
import os

# Remove the empty file
if os.path.exists('/content/Wav2Lip/checkpoints/wav2lip_gan.pth'):
    os.remove('/content/Wav2Lip/checkpoints/wav2lip_gan.pth')

# Download from Google Drive mirror
!gdown "1cyznedRJSeN0hgFMO3zKbsWMrWENnBNE" -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth

size = os.path.getsize('/content/Wav2Lip/checkpoints/wav2lip_gan.pth')/1024/1024
print(f'✅ Model size: {size:.1f} MB')
# Should be around 415 MB

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1cyznedRJSeN0hgFMO3zKbsWMrWENnBNE

but Gdown can't. Please check connections and permissions.


FileNotFoundError: [Errno 2] No such file or directory: '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'

In [10]:
import os
os.makedirs('/content/Wav2Lip/checkpoints', exist_ok=True)

# Try huggingface mirror
!wget -q "https://huggingface.co/numz/wav2lip_studio/resolve/main/Wav2lip/wav2lip_gan.pth" \
    -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth

size = os.path.getsize('/content/Wav2Lip/checkpoints/wav2lip_gan.pth')/1024/1024
print(f'✅ Model size: {size:.1f} MB')

✅ Model size: 415.6 MB


In [11]:
import os
os.chdir('/content/Wav2Lip')

PROJECT_DIR = '/content/drive/MyDrive/supernan'
CLIP_PATH = f'{PROJECT_DIR}/clip_15s.mp4'
HINDI_ALIGNED_PATH = f'{PROJECT_DIR}/hindi_audio_aligned.wav'
OUTPUT_PATH = f'{PROJECT_DIR}/supernan_hindi_dubbed_final.mp4'

!python inference.py \
    --checkpoint_path checkpoints/wav2lip_gan.pth \
    --face "{CLIP_PATH}" \
    --audio "{HINDI_ALIGNED_PATH}" \
    --outfile "{OUTPUT_PATH}" \
    --resize_factor 2 \
    --pads 0 20 0 0 2>&1

if os.path.exists(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH)/1024/1024
    print(f'\n✅ FINAL VIDEO READY: {size:.1f} MB')
else:
    print('\n❌ Output not found')

Using cuda for inference.
Reading video frames...
Number of frames available for inference: 900
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 1202)
Length of mel chunks: 892
  0% 0/7 [00:00<?, ?it/s]
  0% 0/56 [00:00<?, ?it/s]
  2% 1/56 [00:23<21:36, 23.57s/it]
  4% 2/56 [00:26<10:05, 11.21s/it]
  5% 3/56 [00:27<06:07,  6.93s/it]
  7% 4/56 [00:29<04:13,  4.88s/it]
  9% 5/56 [00:31<03:11,  3.75s/it]
 11% 6/56 [00:33<02:35,  3.11s/it]
 12% 7/56 [00:35<02:11,  2.68s/it]
 14% 8/56 [00:37<02:00,  2.52s/it]
 16% 9/56 [00:39<01:52,  2.40s/it]
 18% 10/56 [00:41<01:43,  2.25s/it]
 20% 11/56 [00:43<01:34,  2.11s/it]
 21% 12/56 [00:45<01:29,  2.04s/it]
 23% 13/56 [00:48<01:44,  2.44s/it]
 25% 14/56 [00:50<01:40,  2.38s/it]
 27% 15/56 [00:52<01:33,  2.28s/it]
 29% 16/56 [00:54<01:26,  2.16s/it]
 30%

In [12]:
from google.colab import files
files.download('/content/drive/MyDrive/supernan/supernan_hindi_dubbed_final.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>